<a href="https://colab.research.google.com/github/iniguezd/cartagenaD0aChatGPT/blob/main/dia1/4_ejercicio_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ejercicio final

Escoge un dataset de regresión o clasificación (el que tu quieras) y diseña una red neuronal (de varias capas) para clasificarlo. El algoritmo de entrenamiento debe incluir algún mecanismo que monitorice el loss de validación para la selección del mejor modelo y algún método de regularización (dropout, $L_1$ o $L_2$). La red debe ser evaluada en un conjunto de test.

Si se usa regresión, recomiendo usar el MSE como función de pérdida (https://docs.pytorch.org/docs/stable/generated/torch.nn.MSELoss.html).

Ejemplos clasificación:
- https://archive.ics.uci.edu/dataset/109/wine (se predice `class`)
- https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic (se predice `diagnosis`)

Ejemplos regresión:
- https://www.kaggle.com/datasets/camnugent/california-housing-prices (tiene una variable categórica y una variable con NAs; se predice `median_house_value`)
- https://www.kaggle.com/datasets/heptapod/uci-ml-datasets (se predice `MEDV`)
- https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html

**Nota.** La red tiene que recibir como entrada un valor numérico. Puede ser que haya datasets con features con NAs o con features categóricas. Si hay NAs, se pueden o borrar las filas/columnas o si tenéis conocimientos de machine learning, podéis imputar los valores. Si hay features categóricas, se pueden pasar a valor numérico con una asignación simple (por ejemplo, si la feature es "color" y tiene los valores "rojo", "verde" y "azul", se pueden asignar los valores 0, 1 y 2 respectivamente).


In [1]:
!pip install ucimlrepo


In [3]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [2]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
wine = fetch_ucirepo(id=109)

# data (as pandas dataframes)
X = wine.data.features
y = wine.data.targets

# metadata
print(wine.metadata)

# variable information
print(wine.variables)


{'uci_id': 109, 'name': 'Wine', 'repository_url': 'https://archive.ics.uci.edu/dataset/109/wine', 'data_url': 'https://archive.ics.uci.edu/static/public/109/data.csv', 'abstract': 'Using chemical analysis to determine the origin of wines', 'area': 'Physics and Chemistry', 'tasks': ['Classification'], 'characteristics': ['Tabular'], 'num_instances': 178, 'num_features': 13, 'feature_types': ['Integer', 'Real'], 'demographics': [], 'target_col': ['class'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1992, 'last_updated': 'Mon Aug 28 2023', 'dataset_doi': '10.24432/C5PC7J', 'creators': ['Stefan Aeberhard', 'M. Forina'], 'intro_paper': {'ID': 246, 'type': 'NATIVE', 'title': 'Comparative analysis of statistical pattern recognition methods in high dimensional settings', 'authors': 'S. Aeberhard, D. Coomans, O. Vel', 'venue': 'Pattern Recognition', 'year': 1994, 'journal': None, 'DOI': '10.1016/0031-3203(94)90145-7', 'URL': 'https:

In [17]:
print(X)
print(y)

     Alcohol  Malicacid   Ash  Alcalinity_of_ash  Magnesium  Total_phenols  \
0      14.23       1.71  2.43               15.6        127           2.80   
1      13.20       1.78  2.14               11.2        100           2.65   
2      13.16       2.36  2.67               18.6        101           2.80   
3      14.37       1.95  2.50               16.8        113           3.85   
4      13.24       2.59  2.87               21.0        118           2.80   
..       ...        ...   ...                ...        ...            ...   
173    13.71       5.65  2.45               20.5         95           1.68   
174    13.40       3.91  2.48               23.0        102           1.80   
175    13.27       4.28  2.26               20.0        120           1.59   
176    13.17       2.59  2.37               20.0        120           1.65   
177    14.13       4.10  2.74               24.5         96           2.05   

     Flavanoids  Nonflavanoid_phenols  Proanthocyanins  Color_i

In [22]:
# En PyTorch, los datasets deben heredar de torch.utils.data.Dataset
class WineDataset(Dataset):
    # Inicializamos el dataset cargando los datos
    def __init__(self, features, targets):
        # Convert pandas DataFrames to PyTorch tensors
        self.X = torch.tensor(
            features.values, dtype=torch.float32
        )
        # For classification, targets should be of type torch.long
        # Use .flatten() to ensure y is a 1D tensor
        self.y = torch.tensor(
            targets.values.flatten(), dtype=torch.long
        )

    # PyTorch necesita saber el tamaño del dataset
    def __len__(self):
        return self.X.shape[0]

    # PyTorch necesita poder indexar el dataset
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]